# 🔬 Stage 4: Direct Input (Optical vs Thermal) vs Fusion Validation — Google Colab

This notebook provides the complete GPU-accelerated pipeline to **train, evaluate, and benchmark unimodal direct input approaches (Direct Optical RGB and Direct Thermal IR)** against our **Stage 3 Task-Driven Fusion baseline** (`stage3_gen_best.pt` + `best.pt`) on the official M3FD dataset.

### 🎯 Architectural Objectives:
1. **Unimodal Baselines**: Train standalone YOLOv5su detectors directly on Optical RGB and Thermal IR modalities using identical splits and hyperparameters.
2. **Controlled Benchmark**: Evaluate all three pipelines under identical inference constraints (`imgsz=640`, `batch=16`, `conf=0.001`, `iou=0.6`) on the unseen Test set (840 image pairs).
3. **Automated Decision Matrix**: Empirically quantify mAP@50, mAP@50-95, precision, recall, latency, and throughput to determine whether multimodal fusion provides statistically significant gains over unimodal baselines.
4. **Qualitative Inspection**: Generate 4-column side-by-side grids (RGB vs IR vs Fusion vs GT) for failure case analysis under adverse visual conditions.

### 📁 Google Drive Path Mapping:
- **Project Code Path**: `/content/drive/MyDrive/FYP/code` (or `/content/drive/MyDrive/fyp/code`)
- **M3FD Dataset Archive**: `/content/drive/MyDrive/FYP/M3FD_Detection.zip`

### Step 1: Mount Google Drive & Verify GPU Acceleration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU hardware availability
!nvidia-smi

### Step 2: Install Project Dependencies

In [ ]:
%pip install -q kornia thop tabulate PyYAML tqdm opencv-python matplotlib pandas scipy ultralytics

### Step 3: Fast & Robust Environment Setup

In [ ]:
# @title ⚙️ Step 3: Fast & Robust Environment Setup
import sys
from pathlib import Path

# Dynamic project path resolution
cand_roots = [
    Path('/content/drive/MyDrive/FYP/code'),
    Path('/content/drive/MyDrive/fyp/code'),
    Path('/content/drive/MyDrive/code'),
    Path('/content/code'),
    Path.cwd()
]
CODE_PATH = next((p for p in cand_roots if (p / 'scripts_AG').exists() or (p / 'TarDAL-main').exists()), Path.cwd())

for p in [CODE_PATH, CODE_PATH / 'scripts_AG', CODE_PATH / 'TarDAL-main']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from stage3_colab_setup import setup_stage3_environment
CODE_PATH, ds_root = setup_stage3_environment()


### Step 4: Configure & Train Unimodal Baselines (Direct Optical RGB & Thermal IR)

**Architecture**: Standard YOLOv5su trained directly on raw RGB or IR images using official M3FD 60/20/20 splits.
- **`RESUME = True`**: Automatically resumes interrupted training from the latest checkpoint (`checkpoints/yolov5su_{modality}_last.pt` or `runs/.../last.pt`). If a modality already finished all epochs, it automatically skips to the next!
- **`FREEZE_BACKBONE = False` (Recommended: Full End-to-End Training)**: All 25 layers (0-24) adapt directly to the sensor input.
- **`FREEZE_BACKBONE = True`**: Only the detection head trains; backbone retains COCO weights.

In [ ]:
# @title 🎛️ Step 4: Train Unimodal Baseline Detectors { run: "auto" }
MODALITY = "both" # @param ["both", "rgb", "ir"]
EPOCHS = 15 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}

# --- Independent Model Checkpoint Selection Boxes ---
IR_CHECKPOINT = "best (yolov5su_ir_best.pt - Adaptive LR)" # @param ["best (yolov5su_ir_best.pt - Adaptive LR)", "last (yolov5su_ir_last.pt - Resume)", "scratch (yolov5su.pt)"]
RGB_CHECKPOINT = "best (yolov5su_rgb_best.pt - Adaptive LR)" # @param ["best (yolov5su_rgb_best.pt - Adaptive LR)", "last (yolov5su_rgb_last.pt - Resume)", "scratch (yolov5su.pt)"]

FREEZE_BACKBONE = False # @param {type:"boolean"}
FORCE_TRAIN = False # @param {type:"boolean"}
FORCE_REBUILD_DATASET = False # @param {type:"boolean"}

freeze_val = 10 if FREEZE_BACKBONE else 0
print(f"Target Modality          : {MODALITY}")
print(f"Training Epochs          : {EPOCHS}")
print(f"Batch Size               : {BATCH_SIZE}")
print(f"IR Model Checkpoint      : {IR_CHECKPOINT}")
print(f"RGB Model Checkpoint     : {RGB_CHECKPOINT}")
print(f"Freeze Backbone          : {FREEZE_BACKBONE} (freeze={freeze_val})")
print(f"Force Continue / Retrain : {FORCE_TRAIN}")
print(f"Force Rebuild Dataset    : {FORCE_REBUILD_DATASET}")

def parse_choice(choice_str):
    c = choice_str.lower()
    if 'best' in c:
        return 'best'
    elif 'last' in c:
        return 'last'
    return 'scratch'

ir_opt = parse_choice(IR_CHECKPOINT)
rgb_opt = parse_choice(RGB_CHECKPOINT)

train_cmd = (
    f"python -W ignore {str(CODE_PATH / 'scripts_AG' / '13_train_stage4_unimodal_baselines.py')}"
    f" --modality {MODALITY}"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --freeze {freeze_val}"
    f" --ir_checkpoint {ir_opt}"
    f" --rgb_checkpoint {rgb_opt}"
)
if FORCE_TRAIN:
    train_cmd += " --force_train"
if FORCE_REBUILD_DATASET:
    train_cmd += " --force_rebuild"

print(f"\n>>> {train_cmd}\n")
get_ipython().system(train_cmd)


### Step 5: Tri-Modal Comparative Benchmark & Automated Architectural Decision Matrix

Benchmarks Direct RGB, Direct IR, and Stage 3 Fusion side-by-side on the official Test set (840 pairs) and outputs an automated architectural recommendation.

In [ ]:
# @title 📊 Step 5: Run Tri-Modal Comparative Benchmark & Decision Matrix { run: "auto" }
DATASET_SPLIT = "test" # @param ["test", "val"]
RGB_CHECKPOINT = "yolov5su_rgb_best.pt" # @param ["yolov5su_rgb_best.pt", "best.pt", "default"]
IR_CHECKPOINT = "yolov5su_ir_best.pt" # @param ["yolov5su_ir_best.pt", "best.pt", "default"]
FUSED_DET_CHECKPOINT = "stage3_best.pt" # @param ["stage3_best.pt", "best.pt", "default"]
FUSED_GEN_CHECKPOINT = "stage3_gen_best.pt" # @param ["stage3_gen_best.pt", "tardal-dt.pth", "default"]

eval_script = str(CODE_PATH / 'scripts_AG' / '14_eval_stage4_comparative_benchmark.py')
eval_cmd = f"python -W ignore {eval_script} --split {DATASET_SPLIT}"

ckpt_dir = CODE_PATH / "checkpoints"
if RGB_CHECKPOINT != "default" and (ckpt_dir / RGB_CHECKPOINT).exists():
    eval_cmd += f" --rgb_ckpt {str(ckpt_dir / RGB_CHECKPOINT)}"
if IR_CHECKPOINT != "default" and (ckpt_dir / IR_CHECKPOINT).exists():
    eval_cmd += f" --ir_ckpt {str(ckpt_dir / IR_CHECKPOINT)}"
if FUSED_DET_CHECKPOINT != "default" and (ckpt_dir / FUSED_DET_CHECKPOINT).exists():
    eval_cmd += f" --fused_ckpt {str(ckpt_dir / FUSED_DET_CHECKPOINT)}"
if FUSED_GEN_CHECKPOINT != "default" and (ckpt_dir / FUSED_GEN_CHECKPOINT).exists():
    eval_cmd += f" --gen_ckpt {str(ckpt_dir / FUSED_GEN_CHECKPOINT)}"

print(f"\n>>> {eval_cmd}\n")
get_ipython().system(eval_cmd)

### Step 6: Qualitative Side-by-Side Visual Inspection & Failure Analysis

Generates 4-column comparative visualization grids: Direct RGB vs Direct IR vs Stage 3 Fusion vs Ground Truth.

In [ ]:
# @title 🖼️ Step 6: Generate Qualitative Comparison Visualizations { run: "auto" }
NUM_SAMPLES = 6 # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25 # @param {type:"number"}

viz_script = str(CODE_PATH / 'scripts_AG' / '15_visualize_stage4_comparison.py')
viz_cmd = f"python -W ignore {viz_script} --num_samples {NUM_SAMPLES} --conf {CONFIDENCE_THRESHOLD}"

print(f"\n>>> {viz_cmd}\n")
get_ipython().system(viz_cmd)

# Display generated visualization figures
import glob
from IPython.display import Image, display

viz_dir = CODE_PATH / "runs" / "stage4_comparison_viz"
images = sorted(glob.glob(str(viz_dir / "*.png")))
for img_path in images[:NUM_SAMPLES]:
    print(f"\nVisualizing: {Path(img_path).name}")
    display(Image(filename=img_path))
